# Demo 04. Least Squares and Orthogonal Polynomials

**Standard 4** (least squares), Weeks 3-4.

Interpolation forces the curve through every point; least squares finds the closest fit within a lower-dimensional family, which is appropriate for noisy data. This demo treats the discrete problem, fitting noisy samples with two solvers, and the continuous problem, best $L^2$ approximation of a function, where orthogonal polynomials reduce a coupled system to independent projections.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from ipywidgets import interact, IntSlider, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Orthogonality

The least-squares coefficients solve the normal equations $Gc = b$ with Gram matrix $G_{jk} = \langle \phi_j, \phi_k\rangle$. In the monomial basis on $[-1,1]$, $G$ is the Hilbert matrix, which is severely ill-conditioned. In an orthogonal basis, $\langle P_j, P_k\rangle = 0$ for $j\neq k$, so $G$ is diagonal and each coefficient is an independent projection

$$ c_k = \frac{\langle f, P_k\rangle}{\langle P_k, P_k\rangle}. $$

The discrete analogue is to solve via QR rather than the normal equations.

In [ ]:
# ---------------------------------------------------------------------------
# Discrete least squares: two ways to solve the same problem
# ---------------------------------------------------------------------------
# Given data (x_i, y_i), fit a degree-m polynomial minimising sum (p(x_i)-y_i)^2.
# Stack a Vandermonde matrix V (V[i,j] = x_i^j); we want the coefficients c that
# minimise ||V c - y||_2. Two classic routes:
#
#   (1) Normal equations:  (V^T V) c = V^T y  , simple, but squares the
#       condition number, so it loses accuracy for higher degree.
#   (2) QR factorisation:  V = Q R, solve R c = Q^T y, more stable, the method
#       numerical libraries actually use.

def vandermonde(x, m):
    """Return the (len(x)) x (m+1) Vandermonde matrix for degree m."""
    x = np.asarray(x, float)
    return np.vander(x, m + 1, increasing=True)   # columns 1, x, x^2, ...

def fit_normal_equations(x, y, m):
    """Least-squares polynomial fit via the normal equations (V^T V) c = V^T y."""
    V = vandermonde(x, m)
    return np.linalg.solve(V.T @ V, V.T @ y)

def fit_qr(x, y, m):
    """Least-squares polynomial fit via QR, the numerically stable route."""
    V = vandermonde(x, m)
    Q, R = np.linalg.qr(V)
    return np.linalg.solve(R, Q.T @ y)

def poly_eval(coeffs, x):
    """Evaluate a polynomial given coefficients [c0, c1, ...] (low -> high)."""
    x = np.asarray(x, float)
    return sum(c * x**k for k, c in enumerate(coeffs))

In [ ]:
# ---------------------------------------------------------------------------
# Fit noisy data and compare the conditioning of the two methods
# ---------------------------------------------------------------------------
rng = np.random.default_rng(0)   # fixed seed so the demo is reproducible

def show_fit(degree=3, noise=0.15, n_points=25):
    """Sample a smooth signal + noise, fit by both methods, and report the
    condition number of the normal matrix V^T V (which blows up with degree)."""
    x = np.linspace(-1, 1, n_points)
    truth = np.sin(2 * x)
    y = truth + noise * rng.standard_normal(n_points)

    c_ne = fit_normal_equations(x, y, degree)
    c_qr = fit_qr(x, y, degree)

    xx = np.linspace(-1, 1, 300)
    plt.figure()
    plt.plot(x, y, "b.", ms=8, label="noisy data")
    plt.plot(xx, np.sin(2 * xx), "k-", lw=1, alpha=0.5, label="true signal")
    plt.plot(xx, poly_eval(c_qr, xx), "r-", lw=2, label=f"LS fit (deg {degree})")
    plt.legend(); plt.xlabel("x"); plt.title("Discrete least-squares fit")
    plt.show()

    V = vandermonde(x, degree)
    cond = np.linalg.cond(V.T @ V)
    print(f"condition number of V^T V : {cond:.3e}")
    print(f"||c_normal - c_qr||       : {np.linalg.norm(c_ne - c_qr):.3e}")
    print("(As you raise the degree, cond(V^T V) explodes and the two coefficient"
          " vectors start to disagree, the normal equations are losing digits.)")

show_fit(3, 0.15)

## Continuous least squares (symbolic)

In [ ]:
# ---------------------------------------------------------------------------
# Continuous least squares & orthogonal polynomials  (symbolic, via SymPy)
# ---------------------------------------------------------------------------
# Best L2 approximation of a function on [-1, 1] means minimising the integral
#   integral_{-1}^{1} (f(x) - p(x))^2 dx.
# In the monomial basis this couples all coefficients (an ill-conditioned Hilbert
# matrix). In an orthogonal basis it decouples: each coefficient is an independent
# projection. We build the Legendre polynomials by Gram-Schmidt on {1, x, x^2,...}
# under the inner product <p, q> = integral_{-1}^1 p q dx.

x = sp.symbols("x")

def legendre_basis(n):
    """Return orthogonal polynomials P_0..P_n on [-1,1] via Gram-Schmidt.

    Orthogonalises the monomials 1, x, x^2, ... under <p,q> = int_{-1}^1 p q dx.
    (These are the Legendre polynomials up to normalisation.)
    """
    def inner(p, q):
        return sp.integrate(p * q, (x, -1, 1))
    basis = []
    for k in range(n + 1):
        p = x**k
        for b in basis:                     # subtract projections onto previous
            p = p - inner(x**k, b) / inner(b, b) * b
        basis.append(sp.expand(p))
    return basis

def best_l2_approx(f_expr, n):
    """Best degree-n L2 approximation of f on [-1,1] using the orthogonal basis.

    Because the basis is orthogonal, coefficient k is just <f, P_k>/<P_k, P_k> --
    computed independently, no linear system to solve.
    """
    basis = legendre_basis(n)
    approx = 0
    for b in basis:
        coeff = sp.integrate(f_expr * b, (x, -1, 1)) / sp.integrate(b * b, (x, -1, 1))
        approx += coeff * b
    return sp.expand(approx), basis

# Example: approximate exp(x) by a quadratic and show the orthogonal basis.
approx, basis = best_l2_approx(sp.exp(x), 2)
print("Orthogonal (Legendre) basis P0..P2:")
for k, b in enumerate(basis):
    print(f"  P{k} =", b)
print("\nBest quadratic L2 approximation of exp(x) on [-1,1]:")
sp.pprint(approx)

## Summary

- Least squares minimizes the residual in the 2-norm; the solution satisfies the normal equations.
- Discrete case. The normal equations square the condition number, so QR is preferred. $\operatorname{cond}(V^{\top}V)$ grows with degree and the two coefficient vectors diverge.
- Continuous case. An orthogonal basis diagonalizes the Gram matrix, giving independent projections.
- These orthogonal polynomials reappear in demo 13 for Gaussian quadrature.